# Minnie and V1DD nuclei in CCF space

Overlay randomly sampled Minnie 1822 and V1DD 1196 nucleus positions in the XY, YZ, and XZ planes of CCFv3. These are projections of the same sampled 3D points, not anatomical slices.

First run the base EM ETLs and then [Minnie CCF ETL](../ETL/EM/etl_minnie1822_03_ccf_coord.ipynb) and [V1DD CCF ETL](../ETL/EM/etl_v1dd1196_02_ccf_coord.ipynb), using the same configured output root. This notebook reads their stored `ccfv3_um` SOMA locations and does not transform coordinates or modify CCM tables.

Sample without replacement up to 5,000 nuclei per dataset with seed 42. Each dataset keeps its own cohort; the two projects' cell IDs are not intersected. The same samples, dataset colors, and per-coordinate limits are used in all panels.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from connects_common_connectivity.config import get_settings
from connects_common_connectivity.io import read_reference_spaces, read_spatial_locations
from connects_common_connectivity.io.path_spec import MODEL_TABLE_PATHS
from connects_common_connectivity.models import LocationType, SignedAxis, Unit

In [ ]:
OUTPUT_ROOT = Path(get_settings().output_root)
CCF_REFERENCE_SPACE_ID = "ccfv3_um"
SAMPLE_SIZE = 5_000
RANDOM_STATE = 42
DATASETS = {
    "minnie65": "Minnie 1822",
    "v1dd_1196": "V1DD 1196",
}
COLORS = {"minnie65": "tab:blue", "v1dd_1196": "tab:orange"}
EXPECTED_ORIENTATION = {
    "anterior_to_posterior": SignedAxis.PLUS_X.value,
    "dorsal_to_ventral": SignedAxis.PLUS_Y.value,
    "subject_left_to_right": SignedAxis.PLUS_Z.value,
}
AXIS_LABELS = {
    "x": r"X: anterior $\rightarrow$ posterior ($\mu$m)",
    "y": r"Y: dorsal $\rightarrow$ ventral ($\mu$m)",
    "z": r"Z: subject left $\rightarrow$ right ($\mu$m)",
}
assert isinstance(SAMPLE_SIZE, int) and SAMPLE_SIZE > 0, (
    "SAMPLE_SIZE must be a positive integer."
)
missing_paths = [
    OUTPUT_ROOT / MODEL_TABLE_PATHS[table]
    for table in ("ReferenceSpace", "SpatialLocation")
    if not (OUTPUT_ROOT / MODEL_TABLE_PATHS[table]).is_dir()
]
assert not missing_paths, (
    f"Missing CCM tables: {missing_paths}. Run both EM CCF ETL notebooks first."
)

## read and sample stored nuclei

Require micrometer units and matching anatomical directions in both project-scoped reference spaces. Missing, duplicate, or non-finite coordinates raise errors rather than being silently dropped. Sort cell IDs before sampling so the seed is reproducible even if table row order changes.

In [ ]:
sampled_by_project = {}
for project_id, label in DATASETS.items():
    spaces = read_reference_spaces(
        project_id=project_id,
        reference_space_ids=CCF_REFERENCE_SPACE_ID,
        output_root=OUTPUT_ROOT,
    )
    assert spaces.height == 1, (
        f"Expected one CCF reference space for {label}; run its CCF ETL first."
    )
    space = spaces.row(0, named=True)
    assert space["unit"] == Unit.MICRONS_LENGTH.value, (
        f"Expected CCF micrometer units for {label}."
    )
    orientation = space["anatomical_orientation"]
    assert orientation is not None, f"Missing anatomical orientation for {label}."
    for direction, signed_axis in EXPECTED_ORIENTATION.items():
        assert orientation[direction] == signed_axis, (
            f"Unexpected {direction} axis for {label}."
        )

    locations = read_spatial_locations(
        project_id,
        reference_spaces=CCF_REFERENCE_SPACE_ID,
        location_types=LocationType.SOMA,
        output_root=OUTPUT_ROOT,
    ).select("dataitem_id", "x", "y", "z").sort("dataitem_id")
    assert locations.height > 0, f"No CCF nuclei for {label}; run its CCF ETL first."
    assert locations["dataitem_id"].null_count() == 0, f"Null cell IDs for {label}."
    assert locations["dataitem_id"].n_unique() == locations.height, (
        f"Duplicate CCF nuclei for {label}."
    )
    assert np.isfinite(locations.select("x", "y", "z").to_numpy()).all(), (
        f"Missing or non-finite CCF coordinates for {label}."
    )
    sample = locations.sample(
        n=min(SAMPLE_SIZE, locations.height),
        with_replacement=False,
        shuffle=True,
        seed=RANDOM_STATE,
    )
    sampled_by_project[project_id] = sample
    print(f"{label}: {locations.height:,} stored nuclei; {sample.height:,} sampled")

combined_sample = pl.concat(list(sampled_by_project.values()))
axis_limits = {}
for coordinate in ("x", "y", "z"):
    values = combined_sample[coordinate].to_numpy()
    lower, upper = values.min(), values.max()
    padding = max((upper - lower) * 0.05, 1.0)
    axis_limits[coordinate] = (lower - padding, upper + padding)

## XY, YZ, and XZ overlays

Increasing X points anterior to posterior, Y dorsal to ventral, and Z subject left to right. Labels describe increasing coordinate values, not the viewer's left/right. The XY panel inverts its display Y axis to place dorsal above ventral; the coordinate values themselves remain unchanged. Every panel uses equal physical scaling on both axes.

In [ ]:
planes = (
    ("x", "y", "XY: sagittal projection"),
    ("y", "z", "YZ: coronal projection"),
    ("x", "z", "XZ: horizontal projection"),
)
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for ax, (horizontal, vertical, title) in zip(axes, planes):
    for project_id, label in DATASETS.items():
        sample = sampled_by_project[project_id]
        ax.scatter(
            sample[horizontal].to_numpy(),
            sample[vertical].to_numpy(),
            s=4,
            alpha=0.35,
            color=COLORS[project_id],
            label=f"{label} (n={sample.height:,})",
            linewidths=0,
            rasterized=True,
        )
    ax.set_title(title)
    ax.set_xlabel(AXIS_LABELS[horizontal])
    ax.set_ylabel(AXIS_LABELS[vertical])
    ax.set_xlim(axis_limits[horizontal])
    ax.set_ylim(axis_limits[vertical])
    if vertical == "y":
        ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="box")
    ax.ticklabel_format(style="plain", useOffset=False)
    ax.legend(markerscale=3, frameon=False)
fig.suptitle("Minnie 1822 and V1DD 1196 nuclei in CCFv3")
plt.show()